In [4]:
import sys
import os

# Aller à la racine du projet
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)

Project root added: c:\Users\ghood\Desktop\PF_AML


In [5]:
from app.core.config import AZURE_OPENAI_ENDPOINT

print("Import OK")

Import OK


# Génération de texte avec Azure OpenAI (fonction réutilisable)

Cette fonction encapsule l’appel à Azure OpenAI et permet de générer du texte à partir d’un prompt.
Elle utilise une configuration centralisée pour les credentials et le déploiement.

In [6]:
from app.core.config import (
    AZURE_OPENAI_ENDPOINT,
    AZURE_OPENAI_KEY,
    AZURE_OPENAI_DEPLOYMENT
)

import requests

def generate_text(prompt):

    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_DEPLOYMENT}/chat/completions?api-version=2024-02-15-preview"

    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_KEY
    }

    data = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 100
    }

    response = requests.post(url, headers=headers, json=data)
    result = response.json()

    return result["choices"][0]["message"]["content"]

Traitement intelligent du texte (logique métier)

In [7]:
def process_text(task, text):

    if task == "summarize":
        prompt = f"Résume ce texte en une phrase : {text}"

    elif task == "explain":
        prompt = f"Explique simplement : {text}"

    else:
        return {
            "status": "error",
            "message": f"Tâche '{task}' non supportée"
        }

    try:
        result = generate_text(prompt)

        return {
            "status": "success",
            "task": task,
            "input": text,
            "output": result,
            "source": "azure_openai"
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

In [8]:
print(process_text("summarize", "Azure est une plateforme cloud de Microsoft."))
print(process_text("explain", "Azure permet de déployer des applications."))

{'status': 'success', 'task': 'summarize', 'input': 'Azure est une plateforme cloud de Microsoft.', 'output': 'Azure est le service de cloud computing proposé par Microsoft.', 'source': 'azure_openai'}
{'status': 'success', 'task': 'explain', 'input': 'Azure permet de déployer des applications.', 'output': 'Bien sûr !  \n**Azure** est une plateforme en ligne créée par Microsoft. Elle permet aux entreprises et aux développeurs de mettre en place (« déployer ») des applications sur Internet, sans avoir besoin d’acheter ou gérer leurs propres serveurs. En utilisant Azure, on peut facilement rendre une application accessible à des utilisateurs partout dans le monde, tout en profitant de nombreux outils pour la gérer, la sécuriser et la faire évoluer.', 'source': 'azure_openai'}


Structuration de la réponse (format backend)

In [9]:
def process_text(task, text):
    
    if task == "summarize":
        prompt = f"Résume ce texte en une phrase : {text}"
    
    elif task == "explain":
        prompt = f"Explique simplement : {text}"
    
    else:
        return {
            "status": "error",
            "message": "Tâche non supportée"
        }
    
    result = generate_text(prompt)
    
    return {
        "status": "success",
        "task": task,
        "input": text,
        "output": result,
        "source": "azure_openai"
    }

In [10]:
response = process_text(
    "summarize",
    "Azure est une plateforme cloud de Microsoft qui permet de créer, déployer et gérer des applications."
)

print(response)

{'status': 'success', 'task': 'summarize', 'input': 'Azure est une plateforme cloud de Microsoft qui permet de créer, déployer et gérer des applications.', 'output': "Azure est une plateforme cloud de Microsoft pour le développement et la gestion d'applications.", 'source': 'azure_openai'}


# Point d’entrée intelligent (dispatcher simple)

In [11]:
def analyze(input_data):
    
    data_type = input_data.get("type")
    task = input_data.get("task")
    content = input_data.get("data")

    if data_type == "text":
        return process_text(task, content)

    else:
        return {
            "status": "error",
            "message": f"Type '{data_type}' non supporté"
        }

Test

In [12]:
request = {
    "type": "text",
    "task": "summarize",
    "data": "Azure est une plateforme cloud qui permet de développer des applications."
}

print(analyze(request))

{'status': 'success', 'task': 'summarize', 'input': 'Azure est une plateforme cloud qui permet de développer des applications.', 'output': 'Azure est une plateforme cloud destinée au développement d’applications.', 'source': 'azure_openai'}
